In [52]:
import numpy as np
import inspect
import importlib
import Utils
importlib.reload(Utils)
from rasterstats import zonal_stats
import Constants
importlib.reload(Constants)
import ConstantObjects
importlib.reload(ConstantObjects)
import matplotlib.pyplot as plt

import rasterio
from rasterio.warp import reproject, Resampling





[Line 12] n_cols in ConstantObjects: 21
[Line 14] n_cols: 21, n_rows: 18
[Line 25] n_cols in ConstantObjects: 21
[Line 27] gdf_tree_circles.shape: (378, 1)
x0 =  -10783454.50783975
y0 =  2246774.099946275
draw_grid_box : dx =  5
draw_grid_box : dy =  4
x1,y1 etc
-10783504.354768038 2246750.855941879
-10783527.012462713 2246740.2904853355
-10783520.250570524 2246725.789560743
-10783497.59287585 2246736.3550172867
📍 Center of box: (19.777898, -96.869939)
x0 =  -10783454.50783975
y0 =  2246774.099946275
draw_grid_box : dx =  5
draw_grid_box : dy =  4
x1,y1 etc
-10783477.165534426 2246763.5344897313
-10783499.823229102 2246752.9690331877
-10783479.53755254 2246709.46625941
-10783456.879857862 2246720.0317159537
📍 Center of box: (19.777882, -96.869634)
Constants.n_cols =  21
Constants.n_rows =  18
x0 =  -10783454.50783975
y0 =  2246395.5503548854
draw_grid_box : dx =  5
draw_grid_box : dy =  4
x1,y1 etc
-10783454.50783975 2246395.5503548854
-10782194.50783975 2246395.5503548854
-10782194.50

In [45]:
#dates = Utils.generate_date_range("2025-01-25", "2025-01-26",  "%Y-%m-%d", 5)

#NDMI = Normalized Difference Moisture Index
# Output lists
ndmi_values = []
valid_dates = []

import os
from getpass import getpass
os.environ["ACCESS_TOKEN"] = getpass("Paste CDSE ACCESS_TOKEN (input hidden): ")

#Sanity check, to see if the copernicus token works:
import requests
CAT = "https://catalogue.dataspace.copernicus.eu/odata/v1"
r = requests.get(f"{CAT}/Products", params={"$top":"1","$format":"json"},
                 headers={"Authorization": f"Bearer {os.environ['ACCESS_TOKEN']}",
                          "Accept": "application/json"})
print(r.status_code, r.json().get("value", [{}])[0].get("Name"))


Paste CDSE ACCESS_TOKEN (input hidden):  ········


200 S3A_OL_2_WRR____20161024T183627_20161024T183827_20210511T224724_0119_010_141______MAR_R_NT_003.SEN3


In [44]:
# HTTP requests
import requests

# JSON parser
import json

# XML parser
import xml.etree.ElementTree as ET

# system modules
import os
import re
import sys
import random

# data manipulation
import pandas as pd
import numpy as np

# image manipulation
import rasterio
import matplotlib.pyplot as plt
import matplotlib.image
from rasterio.windows import Window

# file manipulation
from pathlib import Path
# base URL of the product catalogue
catalogue_odata_url = "https://catalogue.dataspace.copernicus.eu/odata/v1"

# search parameters
collection_name = "SENTINEL-2"
#product_type = "S2MSI1C" # level 1C
product_type = "S2MSI2A"  # <-- Level 2A, contains SCL

max_cloud_cover = 1

#lat0 = 19.7782 #location of tree at col 0 row 0. Note there is no actual tree at this location, it is on the driveway.
#lon0 = -96.86942

#origin, right,
aoi = "POLYGON((-96.90 19.75, -96.80 19.75, -96.80 19.85, -96.90 19.85, -96.90 19.75))" #chatgpt

#aoi = "POLYGON((-96.86942  19.7782 ,-96.0 19.7782, -96.0 20.0,-96.86942 20.0, -96.86942  19.7782  ))" #mine
#search_period_start = "2025-01-11T00:00:00.000Z" 
#search_period_end = "2025-01-17T00:00:00.000Z"

#aoi = "POLYGON((20.888443 52.169721,21.124649 52.169721,21.124649 52.271099,20.888443 52.271099,20.888443 52.169721))"
search_period_start = "2025-01-11T00:00:00.000Z"
search_period_end = "2025-01-17T00:00:00.000Z"

search_query = f"{catalogue_odata_url}/Products?$filter=Collection/Name eq '{collection_name}' and Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'productType' and att/OData.CSC.StringAttribute/Value eq '{product_type}') and OData.CSC.Intersects(area=geography'SRID=4326;{aoi}') and ContentDate/Start gt {search_period_start} and ContentDate/Start lt {search_period_end}"

print(f"""\n{search_query.replace(' ', "%20")}\n""")
response = requests.get(search_query).json()

pd.set_option("display.max_colwidth", None)
#print(df[["Id","Name","S3Path"]])

result = pd.DataFrame.from_dict(response["value"])

# print first 3 results
result.head(1000)





https://catalogue.dataspace.copernicus.eu/odata/v1/Products?$filter=Collection/Name%20eq%20'SENTINEL-2'%20and%20Attributes/OData.CSC.StringAttribute/any(att:att/Name%20eq%20'productType'%20and%20att/OData.CSC.StringAttribute/Value%20eq%20'S2MSI2A')%20and%20OData.CSC.Intersects(area=geography'SRID=4326;POLYGON((-96.90%2019.75,%20-96.80%2019.75,%20-96.80%2019.85,%20-96.90%2019.85,%20-96.90%2019.75))')%20and%20ContentDate/Start%20gt%202025-01-11T00:00:00.000Z%20and%20ContentDate/Start%20lt%202025-01-17T00:00:00.000Z



,@odata.mediaContentType,Id,Name,ContentType,ContentLength,OriginDate,PublicationDate,ModificationDate,Online,EvictionDate,S3Path,Checksum,ContentDate,Footprint,GeoFootprint
0,application/octet-stream,568dcebe-06a6-4484-99d8-2d4e9522763a,S2A_MSIL2A_20250112T165651_N0511_R026_T14QQG_20250112T202648.SAFE,application/octet-stream,1103687255,2025-01-12T21:26:50.000000Z,2025-01-12T21:33:29.636166Z,2025-01-12T21:34:46.977870Z,True,9999-12-31T23:59:59.999999Z,/eodata/Sentinel-2/MSI/L2A/2025/01/12/S2A_MSIL2A_20250112T165651_N0511_R026_T14QQG_20250112T202648.SAFE,"[{'Value': '0891d19b57198b772135fccddec7c84e', 'Algorithm': 'MD5', 'ChecksumDate': '2025-01-12T21:34:43.494619Z'}, {'Value': '302ce5857ba5b5410717ee65dca20cf6f9b9c76c6d26393a2010e52417cdc200', 'Algorithm': 'BLAKE3', 'ChecksumDate': '2025-01-12T21:34:45.800984Z'}]","{'Start': '2025-01-12T16:56:51.024000Z', 'End': '2025-01-12T16:56:51.024000Z'}","geography'SRID=4326;POLYGON ((-97.0900750428032 19.88617985844033, -97.10161109124918 18.894435157837528, -96.0600229174343 18.88087538940624, -96.0421758405751 19.871851514748045, -97.0900750428032 19.88617985844033))'","{'type': 'Polygon', 'coordinates': [[[-97.0900750428032, 19.88617985844033], [-97.10161109124918, 18.894435157837528], [-96.0600229174343, 18.88087538940624], [-96.0421758405751, 19.871851514748045], [-97.0900750428032, 19.88617985844033]]]}"
1,application/octet-stream,3c4e2766-304e-43df-8aa7-8ca605e42192,S2A_MSIL2A_20250112T165651_N0511_R026_T14QQH_20250112T202648.SAFE,application/octet-stream,970479537,2025-01-12T21:26:24.000000Z,2025-01-12T21:31:16.374828Z,2025-01-12T21:32:15.326629Z,True,9999-12-31T23:59:59.999999Z,/eodata/Sentinel-2/MSI/L2A/2025/01/12/S2A_MSIL2A_20250112T165651_N0511_R026_T14QQH_20250112T202648.SAFE,"[{'Value': '02c3463a80bd3ce5217a7ad329fc83a1', 'Algorithm': 'MD5', 'ChecksumDate': '2025-01-12T21:32:12.920445Z'}, {'Value': '579315e596af4760556a01e5087fca1068abd9708aebe2b5a3d2fa06df184e94', 'Algorithm': 'BLAKE3', 'ChecksumDate': '2025-01-12T21:32:14.422296Z'}]","{'Start': '2025-01-12T16:56:51.024000Z', 'End': '2025-01-12T16:56:51.024000Z'}","geography'SRID=4326;POLYGON ((-97.07894289730446 20.78948698156559, -97.09113136389475 19.797848807492848, -96.04381002956936 19.783589289270502, -96.02495394736098 20.77445045207495, -97.07894289730446 20.78948698156559))'","{'type': 'Polygon', 'coordinates': [[[-97.07894289730446, 20.78948698156559], [-97.09113136389475, 19.797848807492848], [-96.04381002956936, 19.783589289270502], [-96.02495394736098, 20.77445045207495], [-97.07894289730446, 20.78948698156559]]]}"


In [53]:


# Re-run the previous code after reset
#port Isabel dates

#dates = Utils.generate_date_range("2025-01-02", "2025-06-10",  "%Y-%m-%d", 5)
dates = Utils.generate_date_range("2025-01-02", "2025-01-16",  "%Y-%m-%d", 5)
dates = Utils.generate_date_range("2025-01-12", "2025-01-13",  "%Y-%m-%d", 5)
#dates = Utils.generate_date_range("2025-01-12", "2025-08-15",  "%Y-%m-%d", 5)

print("dates = ",dates)

#NDMI = Normalized Difference Moisture Index
# Output lists
ndmi_values = []
valid_dates = []



# Prepare folder
#os.makedirs("s2_point_series", exist_ok=True)
output_tif_cumulative = "ndmi_cumulative.tif"
output_tif = ""
rgb_tif = ""

# Sample NDMI at the given point
for date in dates:
    print("processing date ",date)
    b02_path = Utils.download_band_dynamic(date, "B02",Constants.lat0,Constants.lon0)  # Blue
    b03_path = Utils.download_band_dynamic(date, "B03",Constants.lat0,Constants.lon0)  # Green
    b04_path = Utils.download_band_dynamic(date, "B04",Constants.lat0,Constants.lon0)  # Red
    b08_path = Utils.download_band_dynamic(date, "B08",Constants.lat0,Constants.lon0)
    b11_path = Utils.download_band_dynamic(date, "B11",Constants.lat0,Constants.lon0)
    QA60_path = Utils.download_band_dynamic(date, "QA60",Constants.lat0,Constants.lon0,"s2_point_series") #Cloud cover
    SCL_path = Utils.download_band_dynamic(date, "SCL",Constants.lat0,Constants.lon0,"s2_point_series") #Cloud cover
    #SCL_path = Utils._download_scl_from_cdse(date,Utils.latlon_to_s2_tile(Constants.lat0,Constants.lon0),filename = os.path.join(cache_dir, f"{mgrs_tile}_{date}_{band_tag}.jp2")


    
    print("b02_path ",  b02_path)
    print("b03_path ",  b03_path)
    print("b04_path ",  b04_path)
    print("b08_path ", b08_path)
    print("b11_path ",  b11_path)


    print(f"[Line {inspect.currentframe().f_lineno}] ... About to try ")
    try:
        with rasterio.open(b08_path) as src_b08, rasterio.open(b11_path) as src_b11 , rasterio.open(b04_path) as src_b04 : # , rasterio.open(b02_path) as src_b02, rasterio.open(b03_path) as src_b03, rasterio.open(b04_path) as src_b04:
            print(f"[Line {inspect.currentframe().f_lineno}] ... ")

            b11_data = src_b11.read(1)
            b04_data = src_b04.read(1)
            resampled_b11 = np.empty((10980, 10980), dtype="float32")
         
            output_tif = Utils.make_path_name("s2_derived_tifs", Utils.latlon_to_s2_tile(Constants.lat0, Constants.lon0),date, "ndmi","tif")
            rgb_float32_tif  = Utils.make_path_name("s2_derived_tifs", Utils.latlon_to_s2_tile(Constants.lat0, Constants.lon0),date, "rgb.float32","tif")
            rgb_uint8_tif    = Utils.make_path_name("s2_derived_tifs", Utils.latlon_to_s2_tile(Constants.lat0, Constants.lon0),date, "rgb.uint8","tif")
            print("About to write ndmi tif")
            Utils.write_ndmi_geotiff(  b08_path, b11_path, output_tif)
            print("About to write float32 rgb tif")
            Utils.write_rgb_geotiff_3857(b02_path, b03_path, b04_path,rgb_float32_tif,"float32")
            print("About to write uint8 rgb tif")

            Utils.write_rgb_geotiff_3857(b02_path, b03_path, b04_path,rgb_uint8_tif, "uint8")

            print("✅ NDMI GeoTIFF in UTM saved to:", output_tif)
            print("✅ RGB uint8 GeoTIFF in UTM saved to:", rgb_uint8_tif)
            print("✅ RGB float32 GeoTIFF in UTM saved to:", rgb_float32_tif)


            #ndmi_3857 = Utils.reproject_UTM_to_3857(output_tif, "ndmi_3857.tif")
            
    
                
    except Exception as e:
        print(f"Skipping {date} due to error: {e}")




dates =  ['2025-01-12']
processing date  2025-01-12
[Line 115] ... found mgrs_tile =  14QQG
[Line 115] ... found mgrs_tile =  14QQG
[Line 115] ... found mgrs_tile =  14QQG
[Line 115] ... found mgrs_tile =  14QQG
[Line 115] ... found mgrs_tile =  14QQG
[Line 115] ... found mgrs_tile =  14QQG
 trying url :  https://sentinel-s2-l1c.s3.amazonaws.com/tiles/14/Q/QG/2025/1/12/0/QA60.jp2
 trying url :  https://sentinel-s2-l1c.s3.amazonaws.com/tiles/14/Q/QG/2025/1/12/1/QA60.jp2
[Line 292] ...       
Failed to download QA60 on 2025-01-12 for tile 14QQG. Tried: https://sentinel-s2-l1c.s3.amazonaws.com/tiles/14/Q/QG/2025/1/12/0/QA60.jp2 (and sequence fallback).
[Line 115] ... found mgrs_tile =  14QQG
[Line 234] ...       
[Line 236] ...       
 trying url :  https://sentinel-s2-l2a.s3.amazonaws.com/tiles/14/Q/QG/2025/1/12/0/R20m/SCL_20m.jp2
 trying url :  https://sentinel-s2-l2a.s3.amazonaws.com/tiles/14/Q/QG/2025/1/12/1/R20m/SCL_20m.jp2
[Line 277] ...       
[Line 347] ...     prod = {'@odata.med

In [5]:
import leafmap
#importlib.reload(ConstantObjects)
print(f"[Line {inspect.currentframe().f_lineno}] ... lat0 = ",Constants.lat0)
print(f"[Line {inspect.currentframe().f_lineno}] ... lon0 = ",Constants.lon0)

m2 = leafmap.Map(basemap="Esri.WorldImagery")
#m2 = leafmap.Map(center=(Constants.lat0,Constants.lon0), zoom=15)
m2.add_raster(output_tif, layer_name="NDMI", opacity=0.9, nodata=float("nan"))

m2.set_center(Constants.lon0, Constants.lat0,  zoom=15)

            
m2.add_gdf(ConstantObjects.gdf_box_terreno_casa, layer_name="Plot Boundary")
m2.add_gdf(ConstantObjects.gdf_box_plots_4_5, layer_name="Plots 4,5")
m2.add_gdf(ConstantObjects.gdf_box_pueblo, layer_name="pueblo")
m2.add_gdf(ConstantObjects.gdf_box_river, layer_name="river")

#show the entire ndmi tile
#m2.add_raster(output_tif_cumulative, layer_name="NDMI", colormap="BrBG", nodata=np.nan, opacity = .55)
#print(f"[Line {inspect.currentframe().f_lineno}] ... b04_path = ",b04_path)
#m2.add_raster(b04_path, layer_name="Red", colormap="BrBG", nodata=np.nan, opacity = .9)



#ndmi_3857 = reproject_to_3857("ndmi_14QQG.tif", "ndmi_14QQG_3857.tif")
#m2.add_raster(output_tif, layer_name="NDMI", opacity=0.9)

#m2.add_raster(ndmi_3857, layer_name="NDMI", opacity=0.9, nodata=float("nan"))


m2

[Line 3] ... lat0 =  19.7782
[Line 4] ... lon0 =  -96.86942


Map(center=[19.7782, -96.86942], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', '…